[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/karzit/temp/blob/master/notebooks/ml-curriculum/02_linear_regression/02_linear_regression.ipynb)

# 02. Linear Regression

> 원본 강의: [Lec 1–4, 모두를 위한 머신러닝과 딥러닝](https://hunkim.github.io/ml/) — ML 개념/용어, Linear Regression, Cost 함수 최소화, 다중 입력 Linear Regression

## 이 장을 배우는 이유

공부 시간과 시험 점수를 함께 적어둔 데이터가 50명분 있다고 해봅시다.
여기서 **"7시간 공부한 사람은 몇 점을 받을까?"** 를 맞히고 싶습니다.

사람이 규칙을 직접 정할 수도 있습니다. "1시간에 2점씩 오른다고 치자." 그런데 왜 하필 2점일까요?
2.3점이 더 맞는 건 아닐까요? 데이터가 50개가 아니라 5만 개이고, 입력이 공부 시간 하나가 아니라
100가지라면 사람이 손으로 정할 수 없습니다.

**데이터를 보고 그 숫자를 컴퓨터가 스스로 정하게 만드는 것**이 머신러닝이고,
그중 가장 단순한 형태가 이번 장의 Linear Regression(선형 회귀)입니다.

이번 장에서 배우는 것

- 예측을 "직선 하나"로 표현하는 방법 ([가설 함수](../../../glossary.md#hypothesis))
- 그 직선이 얼마나 틀렸는지 숫자로 재는 방법 ([비용 함수](../../../glossary.md#cost-function))
- 틀린 정도를 줄여가며 스스로 직선을 고치는 방법 ([경사 하강법](../../../glossary.md#gradient-descent))
- 같은 일을 scikit-learn으로 두 줄에 끝내는 방법

**소요 시간**: 30~40분. 무거운 학습이 없어 모든 셀이 몇 초 안에 끝납니다.

## 이 노트북을 읽는 법

- **셀을 위에서부터 순서대로 실행하세요**(`Shift + Enter`). 아래쪽 셀은 위쪽 셀에서 만든
  변수·함수를 그대로 쓰기 때문에, 중간부터 실행하면 `NameError`가 납니다.
- **실행 결과는 저장되어 있지 않습니다.** 코드 셀 아래가 비어 있는 것이 정상이고,
  직접 실행해야 출력과 그래프가 나타납니다.
- 코드 셀 앞에는 **지금 무엇을 할 것인지**, 뒤에는 **결과를 어떻게 읽는지**를 적어두었습니다.
  결과가 예상과 다르게 나오면 뒤쪽 설명의 "이럴 때는" 부분을 먼저 보세요.
- 낯선 용어는 [glossary.md](../../../glossary.md)에서 찾아보세요.


## 0. 준비 — 필요한 라이브러리 설치

Colab에서 열었다면 아래 셀이 필요한 라이브러리를 설치해줍니다. 내 컴퓨터에서 열었다면
이미 깔려 있을 테니 아무 일도 일어나지 않습니다.

`koreanize-matplotlib`은 그래프에 한글이 □□□로 깨지지 않게 해주는 도구입니다.


In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules
print("Running in Colab:", IN_COLAB)

if IN_COLAB:
    !pip install -q scikit-learn numpy matplotlib koreanize-matplotlib


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 그래프에 한글이 깨지지 않도록 폰트를 설정합니다.
# koreanize-matplotlib이 있으면 그걸 쓰고, 없으면 OS에 설치된 한글 폰트를 찾습니다.
try:
    import koreanize_matplotlib  # noqa: F401
except ImportError:
    import matplotlib.font_manager as fm

    for _name in ["Malgun Gothic", "AppleGothic", "NanumGothic"]:
        if any(_name == f.name for f in fm.fontManager.ttflist):
            plt.rc("font", family=_name)
            break
plt.rcParams["axes.unicode_minus"] = False  # 한글 폰트에서 마이너스 기호가 깨지는 것 방지

print("준비 완료")


## 1. 문제 — 우리가 가진 것과 원하는 것

먼저 데이터를 만듭니다. 실무에서는 CSV 파일을 읽어오지만, 여기서는 **답을 미리 정해놓고**
데이터를 만듭니다. 그래야 학습이 끝난 뒤에 컴퓨터가 제대로 배웠는지 채점할 수 있습니다.

규칙은 `y = 2x + 3`으로 정하고, 여기에 약간의 노이즈를 섞겠습니다. 즉 **정답은 W=2, b=3**입니다.
학습이 끝났을 때 컴퓨터가 이 두 숫자 근처를 찾아내면 성공입니다.


In [ ]:
# x: 공부 시간(0~10시간), y: 시험 점수
# 정답 규칙은 y = 2x + 3 (여기에 사람마다의 편차를 노이즈로 섞는다)
rng = np.random.default_rng(42)   # 42는 시드 — 매번 같은 난수가 나와 결과가 재현된다
x = np.linspace(0, 10, 50)        # linspace(시작, 끝, 개수): 0~10을 50등분한 값들
y = 2 * x + 3 + rng.normal(0, 1.5, size=x.shape)   # normal(평균, 표준편차): 정규분포 노이즈

print(f"데이터 개수: {len(x)}개")
print(f"첫 3명 — 공부 시간: {x[:3].round(2)}, 점수: {y[:3].round(2)}")

plt.scatter(x, y)
plt.xlabel("공부 시간 (x)")
plt.ylabel("시험 점수 (y)")
plt.title("우리가 가진 데이터")
plt.show()


**결과 읽는 법** — 점 50개가 왼쪽 아래에서 오른쪽 위로 퍼져 있습니다.
공부를 많이 할수록 점수가 높지만, 같은 시간을 공부해도 점수는 조금씩 다릅니다(노이즈).

우리가 원하는 것은 **이 점들 한가운데를 지나는 직선 하나**입니다.
그 직선을 찾으면 "7시간 공부하면 몇 점"을 답할 수 있습니다.


## 2. 직선 하나를 숫자 두 개로 — 가설 함수

직선은 숫자 두 개로 완전히 정해집니다.

$$H(x) = Wx + b$$

- $W$ ([가중치](../../../glossary.md#weight-bias), weight): 기울기. **x가 1 늘 때 y가 얼마나 오르는지.**
  여기서는 "1시간 더 공부하면 몇 점 오르는지"입니다.
- $b$ ([편향](../../../glossary.md#weight-bias), bias): y절편. **x가 0일 때의 값.**
  여기서는 "공부를 아예 안 했을 때 받는 점수"입니다.

$H$는 가설(Hypothesis)의 머리글자입니다. "정답은 모르지만 일단 이 직선이라고 해보자"는
뜻이라서 가설이라고 부릅니다.

**그래서 학습이란, 좋은 $W$와 $b$ 한 쌍을 찾는 일**입니다. 다른 건 없습니다.

아무 값이나 넣어보면서 시작해봅시다. $W=1, b=0$ — "1시간에 1점씩 오르고, 안 하면 0점"이라는 가설입니다.


In [ ]:
W_guess, b_guess = 1.0, 0.0        # 아무렇게나 찍은 첫 가설
y_guess = W_guess * x + b_guess    # H(x) = Wx + b 를 그대로 코드로 옮긴 것

plt.scatter(x, y, label="실제 데이터")
plt.plot(x, y_guess, color="red", label=f"내 가설 H(x) = {W_guess}x + {b_guess}")
plt.xlabel("공부 시간 (x)")
plt.ylabel("시험 점수 (y)")
plt.legend()
plt.show()


**결과 읽는 법** — 빨간 직선이 점들보다 한참 아래에 있고, 기울기도 완만합니다.
눈으로 봐도 "틀렸다"는 걸 알 수 있습니다.

문제는 **컴퓨터에게는 눈이 없다**는 것입니다. "한참 아래에 있다"를 컴퓨터가 이해하려면
틀린 정도를 **숫자 하나**로 바꿔줘야 합니다. 그게 다음 절의 주제입니다.


## 3. 얼마나 틀렸는지 재기 — 비용 함수

각 점마다 "예측값 − 실제값"이 오차입니다. 이 오차들을 하나의 숫자로 합쳐야 하는데,
그냥 더하면 안 됩니다. **+5점 틀린 것과 −5점 틀린 것이 상쇄되어 0이 되어버리기 때문**입니다.

그래서 제곱해서 더한 뒤 평균을 냅니다. 이것이 [평균 제곱 오차](../../../glossary.md#mse)(MSE)이고,
선형 회귀에서는 이걸 [비용 함수](../../../glossary.md#cost-function)로 씁니다.

$$J(W, b) = \frac{1}{m}\sum_{i=1}^{m}\left(H(x^{(i)}) - y^{(i)}\right)^2$$

- $m$: 데이터 개수 (여기서는 50)
- 제곱하기 때문에 **크게 틀린 점 하나가 조금 틀린 점 여러 개보다 더 큰 벌점**을 받습니다.
- $J$가 작을수록 좋은 직선입니다.

**학습 = $J(W,b)$를 가장 작게 만드는 $W, b$ 찾기.** 이제 목표가 완전히 숫자로 바뀌었습니다.

정말 그런지 확인해봅시다. 아까 찍은 가설(W=1, b=0)과 정답(W=2, b=3)의 cost를 각각 재봅니다.


In [ ]:
# 주어진 W, b가 데이터를 얼마나 못 맞히는지를 숫자 하나로 돌려주는 함수
def cost(W, b):
    y_pred = W * x + b                 # 이 직선의 예측값
    return np.mean((y_pred - y) ** 2)  # 오차를 제곱해서 평균 (MSE)


print(f"내 가설  (W=1, b=0) 의 cost: {cost(1.0, 0.0):8.3f}")
print(f"정답     (W=2, b=3) 의 cost: {cost(2.0, 3.0):8.3f}")
print(f"엉터리   (W=5, b=0) 의 cost: {cost(5.0, 0.0):8.3f}")


**결과 읽는 법** — 정답에 가까운 (2, 3)의 cost가 가장 작습니다.
cost는 0으로 완전히 떨어지지는 않는데, 데이터에 노이즈가 섞여 있어서
**어떤 직선도 모든 점을 정확히 지나갈 수는 없기 때문**입니다. 그건 실패가 아니라 정상입니다.

이제 목표가 분명해졌습니다. **cost를 가장 작게 만드는 (W, b)를 찾아야 합니다.**
그런데 W와 b를 0.1씩 바꿔가며 전부 시도해볼 수는 없습니다(입력이 100개면 조합이 폭발합니다).
방향을 알고 움직여야 합니다.


## 4. 어느 쪽으로 고칠까 — 경사 하강법

**직관.** 짙은 안개 속 산비탈에 서 있다고 해봅시다. 골짜기 바닥까지 내려가야 하는데
주변이 하나도 보이지 않습니다. 할 수 있는 일은 **발끝으로 지금 서 있는 자리의 경사를 느낀 뒤,
가장 가파르게 내려가는 쪽으로 한 걸음 딛는 것**뿐입니다. 그래도 이걸 반복하면 결국 바닥에 닿습니다.

여기서 "높이"가 cost, "위치"가 (W, b), "경사"가 미분값입니다.
이것이 [경사 하강법](../../../glossary.md#gradient-descent)입니다.

$$W := W - \alpha \frac{\partial J}{\partial W}, \qquad b := b - \alpha \frac{\partial J}{\partial b}$$

- **기울기 앞에 빼기(−)가 붙는 이유**: 기울기는 "올라가는 방향"을 가리킵니다.
  우리는 내려가야 하므로 반대로 갑니다.
- $\alpha$ ([학습률](../../../glossary.md#learning-rate), learning rate): **한 걸음의 보폭**입니다.
  너무 크면 골짜기를 훌쩍 뛰어넘어 반대편 산비탈로 튕겨나가고,
  너무 작으면 도착하기까지 너무 오래 걸립니다.

**미분 결과는 이렇게 생겼습니다.**

$$\frac{\partial J}{\partial W} = \frac{2}{m}\sum (H(x)-y)\cdot x, \qquad
  \frac{\partial J}{\partial b} = \frac{2}{m}\sum (H(x)-y)$$

수식을 외울 필요는 없지만, **왜 이렇게 생겼는지**는 알아두면 코드가 읽힙니다.

- **왜 2가 붙나?** $J$가 오차의 **제곱**이라서입니다. $(\cdot)^2$을 미분하면 지수 2가 앞으로 내려옵니다.
- **왜 $W$쪽에만 $x$를 곱하나?** $H = Wx + b$에서 $W$는 항상 $x$와 곱해져 있습니다.
  그래서 $W$를 조금 바꿨을 때 결과가 흔들리는 정도는 **$x$가 클수록 커집니다**.
  반면 $b$는 그냥 더해지기만 하니 $x$와 상관이 없습니다.
- **왜 $\frac{1}{m}$(평균)으로 나누나?** 데이터가 50개일 때와 5만 개일 때 보폭이
  1000배 달라지면 곤란하기 때문입니다. 평균을 내면 데이터 개수와 상관없이 같은 보폭이 됩니다.

말로만 들으면 와닿지 않으니, **딱 한 걸음만** 직접 내디뎌 봅시다.


In [ ]:
W, b = 1.0, 0.0        # 아까 찍었던 가설에서 출발
lr = 0.01              # 학습률(보폭)
m = len(x)             # 데이터 개수

print(f"[걸음 전] W={W:.3f}, b={b:.3f}, cost={cost(W, b):.3f}")

# 1) 지금 W, b로 예측해본다
y_pred = W * x + b

# 2) 기울기를 구한다 — "W를 키우면 cost가 얼마나 커지는가"
#    오차(y_pred - y)에 x를 곱해 평균 낸 값. 2는 제곱을 미분해서 나온 것.
dW = (2 / m) * np.sum((y_pred - y) * x)
db = (2 / m) * np.sum(y_pred - y)
print(f"          기울기 dW={dW:.3f}, db={db:.3f}")

# 3) 기울기의 반대 방향으로 lr만큼 한 걸음
W = W - lr * dW
b = b - lr * db

print(f"[걸음 후] W={W:.3f}, b={b:.3f}, cost={cost(W, b):.3f}")


**결과 읽는 법** — 두 가지를 확인하세요.

1. **`dW`가 음수**(약 −100)로 나왔습니다. "W를 키우면 cost가 줄어든다"는 뜻이고, 실제로 빼기 연산
   `W - lr * dW` 때문에 W가 1.0에서 **약 2.0으로 커졌습니다**. 정답(W=2) 방향이 맞습니다.
2. **cost가 77에서 10으로 줄었습니다.** 한 걸음이 제대로 작동했다는 증거입니다.

그런데 **`b`는 0에서 0.16으로 거의 움직이지 않았습니다.** 기울기 `db`(약 −16)가
`dW`(약 −100)보다 훨씬 작기 때문입니다. 앞에서 본 것처럼 `dW`에는 x(0~10)가 곱해지지만
`db`에는 곱해지지 않아서, **b는 언제나 W보다 훨씬 느리게 움직입니다.**

그래서 cost도 아직 10 근처입니다. 도달할 수 있는 최선(W=2, b=3일 때 약 1.3)과는 거리가 있습니다.
답은 간단합니다. **같은 걸음을 여러 번 반복하면 됩니다.**


## 5. 같은 걸음을 200번 — 학습 루프

지금부터 방금 한 걸음(예측 → 기울기 → 업데이트)을 200번 반복합니다.
`W=0, b=0`에서 새로 출발해서, 40번마다 중간 상태를 출력해 **cost가 실제로 줄어드는지**
눈으로 확인합니다. 잘 되면 W는 2, b는 3 근처에서 멈춥니다.

`cost_history`는 나중에 그래프를 그리려고 매 걸음의 cost를 적어두는 목록입니다.


In [ ]:
W, b = 0.0, 0.0
lr = 0.01
epochs = 200          # epoch: 데이터 전체를 한 번 훑는 것을 1회로 센다
cost_history = []     # 매 epoch의 cost를 기록해 둔다 (나중에 그래프로)

for epoch in range(epochs):
    y_pred = W * x + b                      # 1) 예측
    c = np.mean((y_pred - y) ** 2)          # 2) 얼마나 틀렸나
    cost_history.append(c)

    if epoch % 40 == 0:
        # 이 시점의 cost는 지금 W, b로 예측했을 때 값이다. 그래서 업데이트 전에 찍는다.
        print(f"epoch {epoch:3d}  cost={c:8.4f}  W={W:.3f}  b={b:.3f}")

    dW = (2 / m) * np.sum((y_pred - y) * x)  # 3) 어느 쪽으로 고쳐야 하나
    db = (2 / m) * np.sum(y_pred - y)

    W -= lr * dW                             # 4) 한 걸음 이동
    b -= lr * db

print(f"\n최종: W={W:.3f}, b={b:.3f}  (정답: W=2, b=3)")

**결과 읽는 법**

- `cost`가 epoch가 늘수록 **계속 줄어들면 정상**입니다. 210 → 2.3 → 1.5로 떨어집니다.
- 최종 `W`는 2.2로 정답(2)에 가깝지만, **`b`는 1.9로 정답(3)에 아직 한참 못 미칩니다.**
  앞 절에서 본 그 이유입니다 — b는 W보다 훨씬 느리게 움직입니다. 200번으로는 부족한 것이고,
  반복을 늘리면 계속 3 쪽으로 갑니다(연습 문제 2번에서 직접 확인합니다).
- W가 정답보다 살짝 큰 2.2인 것도 같은 이야기입니다. b가 모자란 만큼을 W가 대신 메우고 있습니다.

**이럴 때는 이걸 의심하세요.**

| 증상 | 원인 | 해볼 것 |
|---|---|---|
| cost가 오히려 커지고 `nan`이 뜬다 | `lr`이 너무 큼 (골짜기를 뛰어넘음) | `lr = 0.001`로 낮추기 |
| cost가 거의 안 줄어든다 | `lr`이 너무 작음 | `lr = 0.05`로 올리기 |
| cost는 주는데 W가 2 근처에 못 간다 | `epochs`가 부족 | `epochs = 2000`으로 늘리기 |


학습이 잘 됐는지는 **두 그래프로 확인**합니다.

- 왼쪽: epoch마다 cost가 줄어드는 과정 — 계속 내려가야 정상입니다
- 오른쪽: 학습된 직선이 데이터 한가운데를 지나가는지


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))   # subplots(행, 열): 그래프 칸을 격자로 만든다

axes[0].plot(cost_history)
axes[0].set_xlabel("epoch")
axes[0].set_ylabel("cost (MSE)")
axes[0].set_title("Cost가 줄어드는 과정")

axes[1].scatter(x, y, label="데이터")
axes[1].plot(x, W * x + b, color="red", label="학습된 H(x)")
axes[1].set_xlabel("공부 시간 (x)")
axes[1].set_ylabel("시험 점수 (y)")
axes[1].set_title("학습 결과")
axes[1].legend()
plt.show()


**결과 읽는 법** — 왼쪽 곡선이 처음에 가파르게 떨어지다가 점점 평평해집니다.
평평해졌다는 것은 **더 내려갈 곳이 거의 없다**, 즉 골짜기 바닥에 도착했다는 뜻입니다.
오른쪽 빨간 직선은 이제 점들 한가운데를 지나갑니다.

여기까지가 머신러닝의 전부입니다. 앞으로 나올 [로지스틱 회귀](../../../glossary.md#logistic-regression)도, 신경망도, [CNN](../../../glossary.md#cnn)도
**"cost를 정하고 → 기울기를 구해 → 조금씩 고친다"** 라는 이 구조를 그대로 반복합니다.
바뀌는 것은 $H(x)$의 모양과 cost의 종류뿐입니다.


## 6. 입력이 여러 개라면 — 다중 변수 회귀

현실에서 시험 점수를 공부 시간 하나로만 설명할 수는 없습니다.
퀴즈 점수, 중간고사 점수처럼 **입력이 여러 개**인 경우가 보통입니다.

이때는 W도 입력 개수만큼 필요합니다.

$$H(x) = w_1x_1 + w_2x_2 + w_3x_3 + b \;\Longrightarrow\; H(X) = XW + b$$

원리는 똑같습니다. 다만 for문으로 하나씩 곱하면 느리기 때문에 **행렬 곱**으로 한 번에 계산합니다.
여기서 $X$는 (샘플 수 × 특성 수) 행렬, $W$는 (특성 수 × 1) 벡터입니다.

먼저 원본 강의의 Lab("파일 데이터 로딩")처럼 CSV 파일을 하나 만듭니다.
실무에서는 이미 있는 파일을 읽어오지만, 여기서는 읽을 파일부터 직접 만들겠습니다.


In [ ]:
import os

os.makedirs("../../../data", exist_ok=True)

# 퀴즈1, 퀴즈2, 중간고사 점수 -> 기말고사 점수 (입력 3개짜리 회귀 문제)
rng = np.random.default_rng(0)
n_samples = 200        # 학생 수. 적으면 계수 추정이 크게 흔들린다 (연습 문제 3번 참고)
quiz1 = rng.integers(60, 100, n_samples)
quiz2 = rng.integers(60, 100, n_samples)
midterm = rng.integers(60, 100, n_samples)
# 기말 = 퀴즈1의 30% + 퀴즈2의 30% + 중간고사의 40% + 약간의 편차
final = (0.3 * quiz1 + 0.3 * quiz2 + 0.4 * midterm + rng.normal(0, 3, n_samples)).round(1)

data = np.column_stack([quiz1, quiz2, midterm, final])   # 1차원 배열 4개를 열로 붙여 (25, 4) 표로
np.savetxt("../../../data/scores.csv", data, delimiter=",", fmt="%.1f",
           header="quiz1,quiz2,midterm,final", comments="")

print("data/scores.csv 생성 완료")
print(data[:3])   # 첫 3행만 확인 — 열 순서는 quiz1, quiz2, midterm, final


**결과 읽는 법** — 한 행이 학생 한 명입니다. 앞 세 열이 입력(퀴즈1·퀴즈2·중간고사),
마지막 열이 우리가 맞혀야 할 정답(기말)입니다.

이번에도 **답을 미리 정해놓았습니다.** 기말 = 퀴즈1×0.3 + 퀴즈2×0.3 + 중간×0.4입니다.
학습이 끝난 뒤에 나오는 계수가 0.3, 0.3, 0.4 근처인지 보면 됩니다.


### scikit-learn으로 같은 일 하기

지금까지 직접 짠 코드(예측 → cost → 기울기 → 업데이트)를 scikit-learn은 **두 줄**로 처리합니다.

```python
reg = LinearRegression()   # 모델 준비
reg.fit(X_train, y_train)  # 학습 (내부에서 우리가 한 일을 대신 해준다)
```

그러면 왜 앞에서 직접 만들어봤을까요? **`fit()` 안에서 무슨 일이 일어나는지 알기 위해서**입니다.
나중에 학습이 안 될 때 "cost가 안 줄어드는구나", "보폭이 문제구나"를 떠올릴 수 있는 사람과
`fit()`만 아는 사람은 여기서 갈립니다.

새로 나오는 것이 하나 있습니다. `train_test_split`입니다.


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

loaded = np.loadtxt("../../../data/scores.csv", delimiter=",", skiprows=1)  # skiprows=1: 헤더 줄 건너뛰기
X = loaded[:, :3]   # 모든 행의 0~2번 열 = quiz1, quiz2, midterm (입력)
y_multi = loaded[:, 3]    # 3번 열 = final (정답)

# 모델이 답을 통째로 외운 것인지, 진짜로 규칙을 익힌 것인지 확인하려면
# '학습에 쓰지 않은 데이터'가 필요하다. 그래서 20%를 미리 떼어 숨겨둔다.
X_train, X_test, y_train, y_test = train_test_split(X, y_multi, test_size=0.2, random_state=0)
print(f"학습용 {len(X_train)}명 / 평가용 {len(X_test)}명")

reg = LinearRegression()
reg.fit(X_train, y_train)          # 여기서 학습이 일어난다
y_pred_multi = reg.predict(X_test)  # 숨겨뒀던 데이터로 예측

print("\nW (계수):", reg.coef_.round(3), "  <- 정답은 0.3, 0.3, 0.4 근처")
print("b (절편):", round(reg.intercept_, 3))
print("MSE:", round(mean_squared_error(y_test, y_pred_multi), 3))
print("R^2:", round(r2_score(y_test, y_pred_multi), 3))


**결과 읽는 법**

- **계수 3개**가 0.33 / 0.30 / 0.36 정도로, 우리가 심어둔 0.3 / 0.3 / 0.4 근처입니다.
  모델이 규칙을 스스로 찾아냈다는 뜻입니다. 계수는 **"그 입력이 1점 오를 때 기말 점수가 몇 점
  오르는가"** 로 읽습니다. 딱 맞아떨어지지 않는 이유는 데이터에 노이즈를 섞었기 때문이고,
  절편도 정답인 0이 아니라 1 근처로 나옵니다.
- **MSE**는 오차를 제곱해 평균 낸 값이라 **작을수록 좋습니다.**
  단위가 점수²이라서 값의 크기만 봐서는 감이 잘 오지 않습니다.
- **R²(결정계수)** 는 **1에 가까울수록 좋습니다.** 0.77이면 "기말 점수가 흩어진 정도의 77%를
  이 모델이 설명한다"는 뜻입니다. 0이면 그냥 평균값을 찍는 것과 다를 바 없다는 뜻이고,
  음수까지 나올 수 있습니다(평균보다 못하다는 뜻). 나머지 23%는 우리가 일부러 섞은
  노이즈라서, **아무리 좋은 모델을 써도 없앨 수 없습니다.**

**이럴 때는 이걸 의심하세요.** 계수가 0.3 / 0.3 / 0.4에서 크게 벗어난다면 대개 데이터가 부족한
것입니다. `n_samples`를 25로 줄이고 다시 실행해보면 계수가 0.35 / 0.33 / 0.57처럼 엉뚱하게
나오고 절편도 −18까지 튑니다. **데이터가 적으면 모델은 노이즈까지 규칙으로 착각합니다.**
(적은 데이터로도 성능을 믿을 수 있게 재는 방법이 [교차 검증](../../../glossary.md#cross-validation)이고,
`tabular-ml-practice` 03번에서 배웁니다.)


## 정리

이번 장에서 한 일

1. 예측을 직선 하나로 표현했습니다 — $H(x) = Wx + b$
2. 그 직선이 얼마나 틀렸는지 숫자 하나로 쟀습니다 — cost(MSE)
3. 기울기를 구해 cost가 줄어드는 방향으로 조금씩 고쳤습니다 — 경사 하강법
4. 같은 일을 scikit-learn으로 두 줄에 했습니다

**스스로 확인해보기**

- [ ] W와 b가 각각 무엇을 뜻하는지 데이터에 빗대어 말할 수 있다
- [ ] 오차를 그냥 더하지 않고 제곱해서 더하는 이유를 설명할 수 있다
- [ ] 학습률이 너무 클 때와 너무 작을 때 각각 무슨 일이 생기는지 안다
- [ ] cost가 줄지 않을 때 무엇부터 확인할지 안다

## 연습 문제

1. 학습 루프에서 `lr`을 `0.1`, `0.001`로 바꿔 실행하고 `cost_history` 그래프가 어떻게
   달라지는지 비교해보세요. 하나는 발산하고 하나는 너무 느릴 것입니다.
2. `epochs`를 2000으로 늘리면 `b`가 3에 더 가까워지는지 확인해보세요.
3. 6절에서 `n_samples`를 25로 줄여 다시 실행하고, 계수와 절편이 얼마나 흔들리는지 보세요.
   그다음 1000으로 늘리면 어떻게 되는지도 확인해보세요.
4. 6절에서 특성을 하나 더 추가(예: 출석률)해서 성능이 좋아지는지 확인해보세요.

**해설/정답**: [02_linear_regression_solutions.ipynb](02_linear_regression_solutions.ipynb)

다음 노트북([03_classification.ipynb](../03_classification/03_classification.ipynb))에서는
연속값이 아니라 **"합격/불합격" 같은 범주**를 예측합니다. 직선을 그대로 쓰면 왜 안 되는지,
그래서 무엇이 바뀌는지를 같은 흐름으로 따라갑니다.
